In [2]:
#Step-1:- Import Necessary libraries 
import numpy as np
import os

#Step-2:-Load and Explore the data 
train_dir="/kaggle/input/chest-xray-pneumonia/chest_xray/train"
val_dir="/kaggle/input/chest-xray-pneumonia/chest_xray/val"
test_dir="/kaggle/input/chest-xray-pneumonia/chest_xray/test"

#load the data 
categories=["NORMAL","PNEUMONIA"]
for category in categories:
    print(category,":",len(os.listdir(os.path.join(train_dir,category))))

for category in categories:
    print(category,":",len(os.listdir(os.path.join(val_dir,category))))

for category in categories:
    print(category,":",len(os.listdir(os.path.join(test_dir,category))))

#Step-3:-Preprocessing the image
from torchvision import transforms
from torchvision import datasets

train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
])

val_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
])

test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406],std=[0.229,0.224,0.225]),
])

#Use ImageFolder to read the data and label automatically by ImageFolder
train_data=datasets.ImageFolder(train_dir,transform=train_transform)
val_data=datasets.ImageFolder(val_dir,transform=val_transform)
test_data=datasets.ImageFolder(test_dir,transform=test_transform)

from torch.utils.data import DataLoader
#loads and processes the images batches wise to save memory and make training faster
train_loader=DataLoader(train_data,batch_size=32,shuffle=True)
val_loader=DataLoader(val_data,batch_size=32,shuffle=False)
test_loader=DataLoader(test_data,batch_size=32,shuffle=False)

#Step-4:- Device Selection(Cpu/gpu)
import torch
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using Device:- ",device)


#Step-5:- Defining model architecture
from torchvision import models
#Step-5(a) using the pretrained resnet 18 model
model=models.resnet18(pretrained=True)

#Step-5(b) freeze feature extractor, so we do not train whole network by ourself,
#we reuse their learned features
for param in model.parameters():
    param.requires_grad=False

#Step-5(c):- making changes to the pretrained model based on our need
#replace last layer, to add our own custom classification layer
import torch.nn as nn
model.fc=nn.Sequential(
    nn.Linear(model.fc.in_features,256),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(256,1),
    nn.Sigmoid()
)
model=model.to(device)

#Step-6:- Compile the model using Loss function and optimizer
import torch.optim as optim
criterion=nn.BCELoss()
optimizer=optim.Adam(model.fc.parameters(),lr=0.001)

#Step-7:-Training the model
epochs=5
for epoch in range(epochs):
    model.train()
    running_loss=0.0
    for images,labels in train_loader:
        images=images.to(device)
        labels=labels.float().unsqueeze(1).to(device)
        optimizer.zero_grad()
        outputs=model(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer.step()
        running_loss+=loss.item()
    print(f"Epoch [{epoch+1}/{epochs}],Loss: {running_loss/len(train_loader):.4f}")
    
#Step-8:- Evaluate the model
model.eval()
correct=0
total=0
with torch.no_grad():
    for images,labels in test_loader:
        images=images.to(device)
        labels=labels.to(device)
        outputs=model(images)
        predicted=(outputs>0.5).int()
        correct+=(predicted.squeeze() == labels).sum().item()
        total+=labels.size(0)
print("Test accuracy:- ",correct/total)

#Step-9:- Prediction on new images
from PIL import Image
img=Image.open("/kaggle/input/chest-xray-pneumonia/chest_xray/test/PNEUMONIA/person100_bacteria_475.jpeg").convert("RGB")
img=val_transform(img)
img=img.unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    output=model(img)
print("Diagnosis:- ","Pneumonia" if output.item()>0.5 else "Normal")

        

NORMAL : 1341
PNEUMONIA : 3875
NORMAL : 8
PNEUMONIA : 8
NORMAL : 234
PNEUMONIA : 390
Using Device:-  cpu
Epoch [1/5],Loss: 0.2303
Epoch [2/5],Loss: 0.1629
Epoch [3/5],Loss: 0.1407
Epoch [4/5],Loss: 0.1390
Epoch [5/5],Loss: 0.1384
Test accuracy:-  0.8157051282051282


ImportError: cannot import name 'image' from 'PIL' (/usr/local/lib/python3.12/dist-packages/PIL/__init__.py)